# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print('connected')

connected


## 1. My rule and its reason codes

In [16]:
# Rule constants — tune these, but note the change and why in your report.
WINDOW_DAYS       = 15     # length of each comparison window
MIN_PREV_IMPR     = 50     # reliability floor: prior-window impressions needed to be scored
IMPR_DROP_THRESH  = 0.20   # 20%+ impression decline triggers IMPR_DROP
CLICK_DROP_THRESH = 0.20   # 20%+ click decline (beyond the impression decline) triggers CLICK_DROP
POS_SLIP_THRESH   = 1.0    # average position worsening by 1.0+ triggers POSITION_SLIP
ACTIVITY_DROP_FRAC= 0.25   # losing 25%+ of active days triggers ACTIVITY_DROP

# Score weights — must sum to 1.0, each component is already 0-1 before weighting.
W_IMPR, W_CLICK, W_POS, W_ACTIVITY = 0.40, 0.25, 0.20, 0.15
assert abs((W_IMPR + W_CLICK + W_POS + W_ACTIVITY) - 1.0) < 1e-9

REASON_CODES = ['IMPR_DROP', 'CLICK_DROP', 'POSITION_SLIP', 'ACTIVITY_DROP']
print('rule constants set')

rule constants set




**The rule, in plain words:** flag a content item for review when its organic visibility is
falling in a way that's both real and worth a person's time enough prior traffic to trust
the percentage change, and a genuine drop in the most recent 15 days versus the 15 days
before that. The score does **not** claim to know *why* a page is falling seasonality, a
core update, an indexing problem, and a genuine relevance loss all produce the same numbers.
That's why every row carries reason codes instead of a single verdict, and why this is
decision support (a prioritized list for a human to check), not an automatic action.



**Reason codes this rule can emit** (a row can carry more than one):

| Code | Fires when | What it suggests, not proves |
|---|---|---|
| `IMPR_DROP` | impressions fell ≥ 20% vs. the prior 15 days | fewer eyeballs are reaching this page |
| `CLICK_DROP` | clicks fell faster than impressions (CTR compression) | the listing itself looks less clickable — SERP feature change, title/meta staleness |
| `POSITION_SLIP` | average position got meaningfully worse | the drop tracks a ranking loss, not just less search demand |
| `ACTIVITY_DROP` | fewer days with any impressions at all | possible deindexing / technical issue — highest-urgency code |

Note: items with prior window impressions below the reliability floor (`MIN_PREV_IMPR`) are
excluded from scoring entirely at the SQL stage they never reach the `queue` DataFrame, so
there's no `LOW_VOLUME_EXCLUDED` row to see. It's a filter on what gets scored, not a code a
scored row can carry, which is why it's removed from this table.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
windowed = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS decision_day FROM {TABLES['fact_daily']}
    ),
    per_item AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_prev,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_prev,
            AVG(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_last,
            AVG(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_prev,
            COUNT(DISTINCT CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_last,
            COUNT(DISTINCT CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_prev,
            MAX(b.decision_day)                                                      AS decision_day
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.decision_day - INTERVAL ({2*WINDOW_DAYS}) DAY
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT * FROM per_item
    WHERE imp_prev >= {MIN_PREV_IMPR}
""").df()

print(f'{len(windowed):,} content items scored (prior-window impressions >= {MIN_PREV_IMPR})')
windowed.head()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99,893 content items scored (prior-window impressions >= 50)


,client_hash_id,content_hash_id,imp_last,imp_prev,clk_last,clk_prev,pos_last,pos_prev,active_days_last,active_days_prev,decision_day
0,client_e547b89c05043229,content_7ee102e6f6a51610,94.0,117.0,0.0,1.0,21.124118,9.981029,15,15,2026-06-30
1,client_e547b89c05043229,content_830f2ddfc3889c38,591.0,527.0,3.0,3.0,5.059994,6.137864,15,15,2026-06-30
2,client_e547b89c05043229,content_c4a360cce0f93b27,494.0,292.0,3.0,1.0,12.758944,21.351354,15,15,2026-06-30
3,client_e547b89c05043229,content_6d6f08d26a1a1fa1,110.0,68.0,0.0,0.0,60.099348,59.088135,15,15,2026-06-30
4,client_e547b89c05043229,content_763dcb25a959077d,769.0,240.0,7.0,0.0,8.663528,14.527467,15,15,2026-06-30


In [18]:
d = windowed.copy()

# --- component scores, each clipped to [0, 1] ---
d['pct_impr_change']  = (d['imp_last'] - d['imp_prev']) / d['imp_prev']
d['pct_click_change'] = np.where(d['clk_prev'] > 0,
                                  (d['clk_last'] - d['clk_prev']) / d['clk_prev'],
                                  np.nan)
d['pos_change']       = d['pos_last'] - d['pos_prev']            # positive = worse (rank got worse)
d['activity_change']  = (d['active_days_prev'] - d['active_days_last']) / d['active_days_prev'].clip(lower=1)

d['impr_drop_score']     = (-d['pct_impr_change']).clip(lower=0, upper=1)
d['click_drop_score']    = (-d['pct_click_change']).clip(lower=0, upper=1).fillna(0)
d['pos_slip_score']      = (d['pos_change'] / 5.0).clip(lower=0, upper=1)  # 5-position worsening = max score
d['activity_drop_score'] = d['activity_change'].clip(lower=0, upper=1)

d['action_score'] = (
    W_IMPR     * d['impr_drop_score'] +
    W_CLICK    * d['click_drop_score'] +
    W_POS      * d['pos_slip_score'] +
    W_ACTIVITY * d['activity_drop_score']
).round(4)

# --- reason codes: which thresholds actually fired ---
def reasons(row):
    fired = []
    if row['pct_impr_change'] <= -IMPR_DROP_THRESH:
        fired.append('IMPR_DROP')
    if pd.notna(row['pct_click_change']) and row['pct_click_change'] <= -CLICK_DROP_THRESH:
        fired.append('CLICK_DROP')
    if row['pos_change'] >= POS_SLIP_THRESH:
        fired.append('POSITION_SLIP')
    if row['activity_change'] >= ACTIVITY_DROP_FRAC:
        fired.append('ACTIVITY_DROP')
    return fired

d['reason_codes']   = d.apply(reasons, axis=1)
d['n_reasons']       = d['reason_codes'].apply(len)
d['primary_reason']  = d['reason_codes'].apply(lambda r: r[0] if r else 'NONE')
d['reason_codes_str']= d['reason_codes'].apply(lambda r: '|'.join(r) if r else 'NONE')

# Only rows where at least one reason code fired get ranked as an action candidate.
queue = d[d['n_reasons'] > 0].sort_values('action_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)

print(f'{len(queue):,} of {len(d):,} scored items have at least one reason code (candidates for review)')
queue[['rank','client_hash_id','content_hash_id','action_score','primary_reason','reason_codes_str']].head(10)

78,630 of 99,893 scored items have at least one reason code (candidates for review)


,rank,client_hash_id,content_hash_id,action_score,primary_reason,reason_codes_str
0,1,client_08a6a72ff48e62c0,content_ffa6f02528cecf02,0.9882,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
1,2,client_e00b29e582949543,content_e265d60339abbbc1,0.9839,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
2,3,client_62f4a7e64f5e0096,content_1df2769c45c30c9c,0.9820,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
3,4,client_62f4a7e64f5e0096,content_ecca92b845bb006d,0.9816,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
4,5,client_08a6a72ff48e62c0,content_dc75198d687215be,0.9809,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
5,6,client_62f4a7e64f5e0096,content_ed51d167fd32e352,0.9803,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
6,7,client_62f4a7e64f5e0096,content_c78c8cf7de9afbd7,0.9784,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
7,8,client_62f4a7e64f5e0096,content_bbea03cade9a5e12,0.9755,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
8,9,client_08a6a72ff48e62c0,content_f5493341b4824e50,0.9744,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
9,10,client_08a6a72ff48e62c0,content_7eb7dec71101f3ff,0.9708,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP


In [19]:
import os
os.makedirs('work/outputs', exist_ok=True)

out_cols = [
    'rank', 'client_hash_id', 'content_hash_id', 'decision_day',
    'action_score', 'primary_reason', 'reason_codes_str', 'n_reasons',
    'imp_prev', 'imp_last', 'pct_impr_change',
    'clk_prev', 'clk_last', 'pct_click_change',
    'pos_prev', 'pos_last', 'pos_change',
    'active_days_prev', 'active_days_last', 'activity_change',
]
queue[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print('wrote work/outputs/baseline_action_score.csv —', len(queue), 'rows')

wrote work/outputs/baseline_action_score.csv — 78630 rows




Two adjacent 15-day windows ending at the most recent date in the release. Both windows are
in the past relative to decision day, so there's no future-outcome leakage to guard against
the thing to guard against here is *volume* leakage: letting a handful of low-traffic items
dominate the top of the queue just because a tiny denominator makes their percentage swing
huge. That's what `MIN_PREV_IMPR` is for.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = queue.head(20)[[
    'rank','client_hash_id','content_hash_id','action_score',
    'primary_reason','reason_codes_str',
    'imp_prev','imp_last','pct_impr_change',
    'pos_prev','pos_last','pos_change',
]].copy()

# Human judgment fields, differentiated by whether the drop has real volume behind it
# or is a near-floor collapse where a handful of clicks would swing the percentage a lot.
def judge(row):
    if row['imp_prev'] >= MIN_PREV_IMPR * 5:  # comfortably clear of the reliability floor
        return (
            "check indexing / manual audit — prioritize",
            "high — real volume behind this drop, not a floor artifact",
            "a known site-wide event (migration, redirect, outage) explains the drop "
            "and isn't specific to this page"
        )
    else:
        return (
            "check indexing / manual audit",
            "low — near reliability floor, high noise from a small denominator",
            "the drop is seasonal, or a few clicks either way would flip the percentage "
            "back to normal range"
        )

top20[['action', 'confidence', 'would_be_wrong_if']] = top20.apply(
    lambda r: pd.Series(judge(r)), axis=1
)

top20

,rank,client_hash_id,content_hash_id,action_score,primary_reason,reason_codes_str,imp_prev,imp_last,pct_impr_change,pos_prev,pos_last,pos_change,action,confidence,would_be_wrong_if
0,1,client_08a6a72ff48e62c0,content_ffa6f02528cecf02,0.9882,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,226.0,1.0,-0.995575,19.119153,53.000000,33.880847,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
1,2,client_e00b29e582949543,content_e265d60339abbbc1,0.9839,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,74.0,1.0,-0.986486,4.699242,38.000000,33.300758,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
2,3,client_62f4a7e64f5e0096,content_1df2769c45c30c9c,0.9820,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,55.0,1.0,-0.981818,7.128626,81.000000,73.871374,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
3,4,client_62f4a7e64f5e0096,content_ecca92b845bb006d,0.9816,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,52.0,1.0,-0.980769,2.462446,82.000000,79.537554,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
4,5,client_08a6a72ff48e62c0,content_dc75198d687215be,0.9809,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,53.0,1.0,-0.981132,7.906868,37.000000,29.093132,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
5,6,client_62f4a7e64f5e0096,content_ed51d167fd32e352,0.9803,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,89.0,2.0,-0.977528,3.144931,45.000000,41.855069,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
6,7,client_62f4a7e64f5e0096,content_c78c8cf7de9afbd7,0.9784,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,3772.0,2.0,-0.999470,2.067746,76.000000,73.932254,check indexing / manual audit — prioritize,"high — real volume behind this drop, not a flo...","a known site-wide event (migration, redirect, ..."
7,8,client_62f4a7e64f5e0096,content_bbea03cade9a5e12,0.9755,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,386.0,3.0,-0.992228,3.889340,30.000000,26.110660,check indexing / manual audit — prioritize,"high — real volume behind this drop, not a flo...","a known site-wide event (migration, redirect, ..."
8,9,client_08a6a72ff48e62c0,content_f5493341b4824e50,0.9744,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,143.0,2.0,-0.986014,12.803858,54.000000,41.196142,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."
9,10,client_08a6a72ff48e62c0,content_7eb7dec71101f3ff,0.9708,IMPR_DROP,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,68.0,3.0,-0.955882,10.964569,70.000000,59.035431,check indexing / manual audit,"low — near reliability floor, high noise from ...","the drop is seasonal, or a few clicks either w..."


**Top-20 Pattern Analysis**

**Simultaneous reason-code firing:** every item in the top 20 triggers all four reason codes
(`IMPR_DROP`, `CLICK_DROP`, `POSITION_SLIP`, `ACTIVITY_DROP`) simultaneously.

**Underlying cause:** the four warning metrics are reacting to a single shared event total
traffic collapse, where `imp_last` drops to near-zero (1–6 impressions in most rows).

**Where the pattern breaks:** three rows  ranks 7, 8, and 14 (`imp_prev` = 3,772, 386, and
1,301) fire the same four reason codes as everyone else, but with real volume behind the
drop rather than a near-floor collapse. On the reason-code axis they look identical to the
rest of the queue; on the volume axis they're the exception, and Section 4's "Scale
Blindness" finding is exactly why: the score can't currently tell these three apart from the
near-floor rows that make up the other 17.

**Metric Distortion:** When impression counts drop to 1–6 impressions, active days collapse naturally, CTR becomes non-existent, and the average position calculation swings wildly due to small sample sizes average position shifts of 8 to 85 positions show up across the top 20, simply from averaging over a handful of impressions. The four signals are not independent warnings; they are mathematically redundant symptoms of a volume drop near the floor.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Leakage check 1: every column the score used comes from report_date <= decision_day.
# There is no `report_date > decision_day` filter anywhere above, and no column named after
# an outcome — confirm no outcome/label column exists in the scoring frame:
leak_suspects = [c for c in queue.columns if c.startswith('imp_last') is False and
                 ('future' in c.lower() or 'outcome' in c.lower() or 'label' in c.lower())]
print('suspicious outcome-like columns found:', leak_suspects if leak_suspects else 'none')

# --- Leakage check 2: fact_query_90d was deliberately never joined in.
# Week 3 flagged this table's 90-day trailing window as not confirmed to align with an
# arbitrary decision day — same reasoning applies here, so `windowed` and `queue` only ever
# come from fact_daily. Confirm no query_90d column made it into the output frame:
q90d_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share']
leaked_q90d = [c for c in q90d_cols if c in queue.columns]
print('fact_query_90d columns present in scoring frame:', leaked_q90d if leaked_q90d else 'none — correctly excluded')


suspicious outcome-like columns found: none
fact_query_90d columns present in scoring frame: none — correctly excluded


In [22]:
# --- Weak-pick candidates: rows where the percentage swing is being driven by a
# denominator that's barely above the reliability floor — technically passes MIN_PREV_IMPR,
# but a handful of clicks either way would swing the score a lot. Worth a second look.
near_floor = queue[queue['imp_prev'] < MIN_PREV_IMPR * 1.5].sort_values('action_score', ascending=False)
print(f'{len(near_floor):,} candidates within 1.5x of the reliability floor')
near_floor[['rank','client_hash_id','content_hash_id','imp_prev','action_score','primary_reason']].head(10)

9,193 candidates within 1.5x of the reliability floor


,rank,client_hash_id,content_hash_id,imp_prev,action_score,primary_reason
1,2,client_e00b29e582949543,content_e265d60339abbbc1,74.0,0.9839,IMPR_DROP
2,3,client_62f4a7e64f5e0096,content_1df2769c45c30c9c,55.0,0.9820,IMPR_DROP
3,4,client_62f4a7e64f5e0096,content_ecca92b845bb006d,52.0,0.9816,IMPR_DROP
4,5,client_08a6a72ff48e62c0,content_dc75198d687215be,53.0,0.9809,IMPR_DROP
9,10,client_08a6a72ff48e62c0,content_7eb7dec71101f3ff,68.0,0.9708,IMPR_DROP
10,11,client_62f4a7e64f5e0096,content_a70ee3c42f7b38e7,54.0,0.9702,IMPR_DROP
12,13,client_73cda7b4e4f265ea,content_b80aecaf4bf66395,63.0,0.9687,IMPR_DROP
14,15,client_73cda7b4e4f265ea,content_bc6c904c6947a1ca,58.0,0.9678,IMPR_DROP
15,16,client_0b245132bb722950,content_640da2418197124e,60.0,0.9636,IMPR_DROP
17,18,client_62f4a7e64f5e0096,content_eed7a183b43db9bc,72.0,0.9603,IMPR_DROP


In [23]:
# --- Weak-pick candidates: one client dominating the top of the queue.
# If a single client_hash_id is most of the top 20, the score may just be tracking that
# client's overall size, not a genuinely worse decline.
client_share_top20 = queue.head(20)['client_hash_id'].value_counts(normalize=True).round(2)
print('client share of top 20 rows:')
client_share_top20

client share of top 20 rows:


,proportion
client_hash_id,
client_62f4a7e64f5e0096,0.45
client_08a6a72ff48e62c0,0.25
client_23a62021009f63c4,0.10
client_73cda7b4e4f265ea,0.10
client_e00b29e582949543,0.05
client_0b245132bb722950,0.05


Weak Picks & Score Limitations AnalysisScale Blindness (Volume Invariance):

**Scale Blindness (Volume Invariance):** The baseline formula ranks items based purely on
percentage drops and does not account for absolute volume loss. For example, Rank 4
(imp_prev = 52) scores 0.9816, while Rank 7 (imp_prev = 3,772) scores 0.9784  nearly
identical scores despite a 70x difference in scale. In a real business queue, losing 3,770
impressions per window is a much higher priority than losing 51 impressions near the floor,
but the capped percentage formula treats them almost the same.

**Client Concentration Bias:** Two client IDs dominate the top 20: client_62f4a7e64f5e0096 accounts for 45% (9 of 20 rows), and client_08a6a72ff48e62c0 accounts for 25% (5 of 20 rows), totaling 70% of the top queue. This indicates the queue may be reflecting overall client catalog scale or a broad site-wide event rather than isolated content degradation.

**Leakage verification:** Checked and confirmed zero leakage. All inputs are strictly derived from past performance within `fact_daily` prior to `decision_day`. No future-window data or `fact_query_90d` columns were joined or referenced during scoring.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.